# 03 - Apply Iceberg SCD Type 2

Run after notebook 02 has stopped Spark. Start a fresh Spark-enabled kernel and run all cells in order. Apply each unprocessed product event in sequence: expire the old version, insert a changed version, then record the event in a ledger.

Version keys are deterministic CDC IDs. Replays skip applied events. Timestamp-only updates do not create a version. Deletes expire the current version; reinserts create a new lifetime. This small, single-writer example uses several Iceberg commits per event. Read results after processing completes; it is not an atomic multi-table transaction.


In [ ]:
import os
import re
from pathlib import Path

DB = os.getenv("SCD_DATABASE", "scd_workshop")
assert re.fullmatch(r"[a-z][a-z0-9_]{0,40}", DB)
CAPTURE_MODE = "replay"  # Use "live" after capturing your own source changes.
assert CAPTURE_MODE in ("replay", "live")
CAPTURE_ROOT = Path.cwd() / ("captured-binlog" if CAPTURE_MODE == "replay" else "lab_data") / DB
LAB_DIR = Path(os.getenv("SCD_CAPTURE_DIR", str(CAPTURE_ROOT)))
LAB_DIR.mkdir(parents=True, exist_ok=True)
BRONZE = f"s3a://warehouse/scd-capture/{DB}"
WAREHOUSE = f"s3://warehouse/iceberg/{DB}"
print("Database / namespace:", DB)
print("Captured batches:", LAB_DIR / "batches")


In [ ]:
import os

import socket

REQUIRED_ENV = ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "POLARIS_CLIENT_ID", "POLARIS_CLIENT_SECRET"]

def preflight():
    missing = [name for name in REQUIRED_ENV if not os.environ.get(name)]
    if missing:
        raise RuntimeError("Missing existing credential environment names: " + ", ".join(missing))
    import importlib.util
    from pathlib import Path
    if importlib.util.find_spec("pyspark") is None:
        raise RuntimeError("Choose the existing Spark-enabled Python kernel; do not pip install PySpark")
    jars = Path(os.environ.get("SPARK_HOME", "/opt/spark")) / "jars"
    patterns = ["iceberg-spark-runtime-*.jar", "iceberg-aws-bundle-*.jar",
                "hadoop-aws-*.jar", "aws-java-sdk-bundle-*.jar"]
    absent = [pattern for pattern in patterns if not list(jars.glob(pattern))]
    if absent:
        raise RuntimeError("Missing image JARs; stop without rebuilding/installing: " + ", ".join(absent))
    print("PREFLIGHT_PASS: required environment names, PySpark and image JARs present")

def start_spark(app_name):
    preflight()
    from pyspark.sql import SparkSession
    if SparkSession.getActiveSession() is not None:
        raise RuntimeError("Stop the existing Spark session and restart this kernel before setup")
    builder = (SparkSession.builder.appName(app_name).master("spark://spark-master:7077")
        .config("spark.driver.host", os.environ.get("POD_IP") or socket.gethostbyname(socket.gethostname()))
        .config("spark.driver.bindAddress", "0.0.0.0")
        .config("spark.cores.max", "1").config("spark.executor.cores", "1")
        .config("spark.executor.memory", "512m").config("spark.sql.shuffle.partitions", "2")
        .config("spark.sql.session.timeZone", "UTC")
        .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
        .config("spark.sql.catalog.polaris", "org.apache.iceberg.spark.SparkCatalog")
        .config("spark.sql.catalog.polaris.type", "rest")
        .config("spark.sql.catalog.polaris.uri", os.environ.get("POLARIS_URI", "http://polaris:8181/api/catalog"))
        .config("spark.sql.catalog.polaris.warehouse", "training")
        .config("spark.sql.catalog.polaris.rest.auth.type", "oauth2")
        .config("spark.sql.catalog.polaris.scope", "PRINCIPAL_ROLE:ALL")
        .config("spark.sql.catalog.polaris.oauth2-server-uri", os.environ.get("POLARIS_URI", "http://polaris:8181/api/catalog").rstrip("/") + "/v1/oauth/tokens")
        .config("spark.sql.catalog.polaris.header.Polaris-Realm", "POLARIS")
        .config("spark.sql.catalog.polaris.rest-metrics-reporting-enabled", "false")
        .config("spark.sql.catalog.polaris.credential", os.environ["POLARIS_CLIENT_ID"] + ":" + os.environ["POLARIS_CLIENT_SECRET"])
        .config("spark.sql.catalog.polaris.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
        .config("spark.sql.catalog.polaris.s3.endpoint", os.environ.get("S3_ENDPOINT", "http://seaweedfs:8333"))
        .config("spark.sql.catalog.polaris.s3.path-style-access", "true")
        .config("spark.sql.catalog.polaris.s3.access-key-id", os.environ["AWS_ACCESS_KEY_ID"])
        .config("spark.sql.catalog.polaris.s3.secret-access-key", os.environ["AWS_SECRET_ACCESS_KEY"])
        .config("spark.sql.catalog.polaris.client.region", "us-east-1")
        .config("spark.hadoop.fs.s3a.endpoint", os.environ.get("S3_ENDPOINT", "http://seaweedfs:8333"))
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
        .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
        .config("spark.hadoop.fs.s3a.access.key", os.environ["AWS_ACCESS_KEY_ID"])
        .config("spark.hadoop.fs.s3a.secret.key", os.environ["AWS_SECRET_ACCESS_KEY"])
        .config("spark.redaction.regex", "(?i)secret|password|token|access.?key|credential"))
    spark = builder.getOrCreate()
    spark.sparkContext.setLogLevel("WARN")
    print("SPARK_STARTED", spark.version, spark.sparkContext.applicationId)
    return spark

from pyspark.sql import Window, functions as F
spark = start_spark("scd-" + DB)
spark.sql(f"CREATE NAMESPACE IF NOT EXISTS polaris.{DB} LOCATION '{WAREHOUSE}'")


## History table and a tiny applied-event ledger

`product_sk = cdc_id` identifies a version deterministically. Track name, category, color and
list price; changing only `updated_at` creates no version. A DELETE closes the current
version without inserting a tombstone. A later INSERT starts a new lifetime.

Validity intervals are half-open: **[valid_from, valid_to)**. NULL valid_to means current.
The ledger prevents replaying completed events; deterministic keys also handle retrying an
event interrupted between expiration, insertion and recording completion. There is no
cross-table transaction: do not read partially processed results or run concurrent writers.

In [ ]:
assert spark.catalog.tableExists(f"polaris.{DB}.product_changes"), "Run notebook 2 first."
spark.sql(f"""CREATE TABLE IF NOT EXISTS polaris.{DB}.dim_product_scd2 (
    product_sk STRING, product_id BIGINT, product_name STRING, category STRING, color STRING,
    list_price DECIMAL(10,2), valid_from TIMESTAMP, valid_to TIMESTAMP, is_current BOOLEAN)
    USING iceberg""")
spark.sql(f"""CREATE TABLE IF NOT EXISTS polaris.{DB}.scd2_applied_events (
    cdc_id STRING, sequence BIGINT, batch_id BIGINT) USING iceberg""")
pending = spark.sql(f"""SELECT s.* FROM polaris.{DB}.product_changes s
    LEFT ANTI JOIN polaris.{DB}.scd2_applied_events p ON s.cdc_id=p.cdc_id
    ORDER BY s.sequence""")
pending.show(100, truncate=False)

## Expire the previous version, then insert the new version

`<=>` is Spark SQL's null-safe equality: two NULL categories count as unchanged.
For each INSERT/UPDATE, compare tracked attributes to the current row. If unchanged, skip
both history writes. If changed, expire then insert. A retry after expiration still sees
the new version is missing; a retry after insertion sees identical current attributes.
For DELETE, simply expire. Record completion only after the history writes succeed.

In [ ]:
source_schema = pending.schema
for change in pending.collect():  # Tiny training dataset; not for large production CDC.
    spark.createDataFrame([change], source_schema).createOrReplaceTempView("one_change")
    if change.operation == "D":
        spark.sql(f"""MERGE INTO polaris.{DB}.dim_product_scd2 t USING one_change s
            ON t.product_id=s.product_id AND t.is_current=true
            WHEN MATCHED THEN UPDATE SET t.valid_to=s.event_time, t.is_current=false""")
    else:
        changed = spark.sql(f"""SELECT s.* FROM one_change s
            LEFT JOIN polaris.{DB}.dim_product_scd2 t ON s.product_id=t.product_id AND t.is_current=true
            WHERE t.product_sk IS NULL OR NOT (
                t.product_name <=> s.product_name AND t.category <=> s.category
                AND t.color <=> s.color AND t.list_price <=> s.list_price)""").count() > 0
        if changed:
            spark.sql(f"""MERGE INTO polaris.{DB}.dim_product_scd2 t USING one_change s
                ON t.product_id=s.product_id AND t.is_current=true AND t.product_sk<>s.cdc_id
                WHEN MATCHED THEN UPDATE SET t.valid_to=s.event_time, t.is_current=false""")
            spark.sql(f"""MERGE INTO polaris.{DB}.dim_product_scd2 t USING one_change s
                ON t.product_sk=s.cdc_id
                WHEN NOT MATCHED THEN INSERT
                    (product_sk,product_id,product_name,category,color,list_price,valid_from,valid_to,is_current)
                    VALUES(s.cdc_id,s.product_id,s.product_name,s.category,s.color,s.list_price,s.event_time,NULL,true)""")
    spark.sql(f"""MERGE INTO polaris.{DB}.scd2_applied_events t USING one_change s
        ON t.cdc_id=s.cdc_id WHEN NOT MATCHED THEN INSERT (cdc_id,sequence,batch_id)
        VALUES(s.cdc_id,s.sequence,s.batch_id)""")
    print("Applied", change.sequence, change.operation, "product", change.product_id)

## Compare current state with preserved history

After Change 4, product 1 has **one SCD1 row** and **five SCD2 versions**:
Premium/Black/100 → Premium/Black/95 → Premium/Silver/95 → Standard/Silver/95 → Standard/Silver/90.
Change 5 must leave those five versions unchanged. Change 6 expires product 3; Change 7
creates a second, current lifetime. Source CDC timestamps determine validity boundaries.

In [ ]:
spark.table(f"polaris.{DB}.dim_product_scd1").orderBy("product_id").show(truncate=False)
spark.table(f"polaris.{DB}.dim_product_scd2").orderBy("product_id", "valid_from").show(100, truncate=False)
spark.table(f"polaris.{DB}.latest_orders").orderBy("order_id").show(truncate=False)

## Checks that hold after every change

Verify unique versions, at most one current row, valid nonoverlapping intervals, agreement
between current SCD1/SCD2 attributes and complete event processing. Rerun notebooks 2 and 3:
counts and history should not change. These assertions also work before the final change.

In [ ]:
assert spark.sql(f"""SELECT product_sk FROM polaris.{DB}.dim_product_scd2
    GROUP BY product_sk HAVING COUNT(*)>1""").count() == 0
assert spark.sql(f"""SELECT product_id FROM polaris.{DB}.dim_product_scd2 WHERE is_current
    GROUP BY product_id HAVING COUNT(*)>1""").count() == 0
assert spark.sql(f"""SELECT * FROM polaris.{DB}.dim_product_scd2
    WHERE (is_current AND valid_to IS NOT NULL) OR (NOT is_current AND valid_to IS NULL)
       OR valid_to <= valid_from""").count() == 0
assert spark.sql(f"""SELECT * FROM (
    SELECT *, LEAD(valid_from) OVER(PARTITION BY product_id ORDER BY valid_from) AS next_from
    FROM polaris.{DB}.dim_product_scd2)
    WHERE next_from IS NOT NULL AND (valid_to IS NULL OR valid_to > next_from)""").count() == 0
attrs = ["product_id", "product_name", "category", "color", "list_price"]
scd1 = spark.table(f"polaris.{DB}.dim_product_scd1").select(*attrs)
scd2 = spark.table(f"polaris.{DB}.dim_product_scd2").filter("is_current").select(*attrs)
assert scd1.exceptAll(scd2).count() == 0 and scd2.exceptAll(scd1).count() == 0
assert spark.sql(f"""SELECT c.cdc_id FROM polaris.{DB}.product_changes c
    LEFT ANTI JOIN polaris.{DB}.scd2_applied_events p ON c.cdc_id=p.cdc_id""").count() == 0
print("PASS: current state, version uniqueness, interval validity and CDC coverage")

## Final scenario checks (after Change 4 or later)

This cell waits until the final price change has arrived. Late PAID must not replace
DELIVERED for order 1001. Order 1006 checks the equal-business-time tie-break.

In [ ]:
final = spark.table(f"polaris.{DB}.dim_product_scd1").filter("product_id=1 AND list_price=90").count() == 1
if final:
    versions = spark.sql(f"""SELECT category,color,list_price FROM polaris.{DB}.dim_product_scd2
        WHERE product_id=1 ORDER BY valid_from""").collect()
    assert [(r.category,r.color,str(r.list_price)) for r in versions] == [
        ("Premium","Black","100.00"), ("Premium","Black","95.00"),
        ("Premium","Silver","95.00"), ("Standard","Silver","95.00"),
        ("Standard","Silver","90.00")]
    states = {r.order_id:r.status for r in spark.table(f"polaris.{DB}.latest_orders").collect()}
    assert {k:states[k] for k in [1001,1002,1003,1006,1009]} == {
        1001:"DELIVERED",1002:"CANCELLED",1003:"PAID",1006:"PAID",1009:"CREATED"}
    print("PASS: five product versions, late event, timestamp tie-break and mixed order states")
else:
    print("Continue source changes through Change 4 for final scenario checks.")
spark.sql(f"""SELECT selling_price, COUNT(*) AS orders, SUM(quantity) AS quantity,
    SUM(quantity*selling_price) AS booked_value FROM polaris.{DB}.latest_orders
    WHERE status IN ('PAID','DELIVERED') GROUP BY selling_price ORDER BY selling_price DESC""").show(truncate=False)

## Discussion

* CDC answers **what changed** in MySQL; SCD1 keeps current attributes; SCD2 keeps their history.
* Source sequence orders product changes. Business `event_time`, then `event_id`, orders order events.
* An order's actual selling price remains unchanged when today's product list price changes.
* Revisit notebook 1 for a new change, then run 2 → 3. For a separate dataset, change DB in all three.

References: [Iceberg 1.10 Spark writes](https://iceberg.apache.org/docs/1.10.0/spark-writes/),
[Hadoop catalog configuration](https://iceberg.apache.org/docs/1.10.0/spark-configuration/),
[MySQL Python binlog reader](https://github.com/julien-duponchelle/python-mysql-replication).

## Record table contents for replay comparison

Run twice with the same input. Each row count and SHA-256 fingerprint must match, including historical validity intervals.

In [ ]:
import hashlib
import json
summary = {}
for table in ["product_changes", "order_history", "latest_orders", "dim_product_scd1", "dim_product_scd2", "scd2_applied_events"]:
    rows = sorted(spark.table(f"polaris.{DB}.{table}").toJSON().collect())
    summary[table] = {"rows": len(rows), "sha256": hashlib.sha256("\n".join(rows).encode()).hexdigest()}
print("PASS RESULT_FINGERPRINT " + json.dumps(summary, sort_keys=True))


## Release Spark

Stop this session before starting the next notebook.

In [ ]:
spark.stop()
